Cleaning the B3DB dataset

In [ ]:
import pandas as pd
import numpy as np
import itertools
from rdkit import Chem
from rdkit.Chem import AllChem
import networkx as nx
import seaborn as sns
import matplotlib.pyplot as plt
from feature_selection import structural_similarity, mw_diff

In [ ]:
input_file = '../datasets/b3db_initial.csv'
output_file = '../datasets/b3db_drug_like.csv'

In [ ]:
# Import the removed metals dataset – removed manually:
full_dataset = pd.read_csv(input_file)
smiles = full_dataset['SMILES']

# Check validity by generating fp using rdkit.
invalid = []
fpgen = AllChem.GetRDKitFPGenerator()
for smile_id, smile in enumerate(smiles): 
    try: 
        mol = Chem.MolFromSmiles(smile)
        fpgen.GetFingerprint(mol)
    except: 
        invalid.append(smile_id)

# Remove invalid structures. 
all_valid_dataset = full_dataset.drop(invalid, axis=0)
all_valid_dataset = all_valid_dataset.reset_index()
smiles_valid = all_valid_dataset['SMILES']
logBB_valid = all_valid_dataset['logBB']
bbb_class_valid = all_valid_dataset['Class']

print('Dataset size after removing metals and unstandardizable smiles structures: ', len(all_valid_dataset))

# Calculate structural similarity and MW to identify stereoisomers.
sim_vals, _, _ = structural_similarity(smiles_valid)
sim_matrix_valid = np.array(sim_vals).reshape((len(smiles_valid), len(smiles_valid)))
mw_diff_matrix = np.array(mw_diff(smiles_valid)).reshape((len(smiles_valid), len(smiles_valid)))

sim_inds, mw_inds= np.column_stack(np.where(sim_matrix_valid == 1)),  np.column_stack(np.where(mw_diff_matrix == 0))
sim_inds, mw_inds = np.array([np.array([x[0], x[1]]) for x in sim_inds if x[0]<x[1]]), np.array([np.array([x[0], x[1]]) for x in mw_inds if x[0]<x[1]])
drop_inds = [ list(el) for el in sim_inds if el in mw_inds]

# Find clusters of stereoisomers and select the one with the lowest index (most likely to have a logBB value)
G = nx.Graph()
G.add_edges_from(drop_inds)
clusters = list(nx.connected_components(G))
drop_ind_clusters = [sorted(list(c))[1:] for c in clusters]
drop_ind_clusters_flat = [item for sublist in drop_ind_clusters for item in sublist]

# Remove all redundant stereoisomers
all_valid_dataset_no_str = all_valid_dataset.drop(drop_ind_clusters_flat, axis=0)
all_valid_dataset_no_str = all_valid_dataset_no_str.reset_index()

print('Dataset size after removal of metals, unstandardizable structures, and stereoisomers: ', len(all_valid_dataset_no_str))

all_valid_dataset_drug_like = all_valid_dataset_no_str[all_valid_dataset_no_str["MW"]<=800]
all_valid_dataset_drug_like = all_valid_dataset_drug_like[all_valid_dataset_drug_like['nHBDon']<=6]
all_valid_dataset_drug_like = all_valid_dataset_drug_like[all_valid_dataset_drug_like['nHBAcc']<=11]
all_valid_dataset_drug_like = all_valid_dataset_drug_like[all_valid_dataset_drug_like['SLogP']<=5]
all_valid_dataset_drug_like = all_valid_dataset_drug_like[all_valid_dataset_drug_like['SLogP']>0]
all_valid_dataset_drug_like = all_valid_dataset_drug_like[all_valid_dataset_drug_like['TopoPSA']<=180]

all_valid_dataset_drug_like.to_csv(output_file)

print('Dataset size after removal of metals, unstandardizable structures, stereoisomers, and non-druglike molecules: ', len(all_valid_dataset_drug_like))


[13:50:30] Explicit valence for atom # 10 C, 4, is greater than permitted
[13:50:36] Explicit valence for atom # 10 C, 4, is greater than permitted


Dataset size after removing metals and unstandardizable smiles structures:  7793
Dataset size after removal of metals, unstandardizable structures, and stereoisomers:  4004
Dataset size after removal of metals, unstandardizable structures, stereoisomers, and non-druglike molecules:  3117
